In [0]:
%sh
rm -r /Workspace/Users/josepaulrichard@gmail.com/device_stream_new
mkdir /Workspace/Users/josepaulrichard@gmail.com/device_stream_new
wget -O /Workspace/Users/josepaulrichard@gmail.com/device_stream_new/device_data.csv https://raw.githubusercontent.com/Kiran-255666/datasets/refs/heads/main/device_data.csv

--2026-05-22 10:52:26--  https://raw.githubusercontent.com/Kiran-255666/datasets/refs/heads/main/device_data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5841 (5.7K) [text/plain]
Saving to: ‘/Workspace/Users/josepaulrichard@gmail.com/device_stream_new/device_data.csv’

     0K .....                                                 100% 10.9M=0.001s

2026-05-22 10:52:26 (10.9 MB/s) - ‘/Workspace/Users/josepaulrichard@gmail.com/device_stream_new/device_data.csv’ saved [5841/5841]



In [0]:
%python

from pyspark.sql.functions import *
from pyspark.sql.types import *

# ============================================================
# USE CATALOG + SCHEMA
# ============================================================

spark.sql("USE CATALOG pikapika")
spark.sql("USE SCHEMA default")

# ============================================================
# DROP OLD TABLES
# ============================================================

spark.sql("DROP TABLE IF EXISTS pikapika.default.device_bronze_jprg")
spark.sql("DROP TABLE IF EXISTS pikapika.default.device_silver_jprg")
spark.sql("DROP TABLE IF EXISTS pikapika.default.device_gold_jprg")

print("Old tables dropped")

# ============================================================
# READ SOURCE CSV
# ============================================================

inputPath = "/Workspace/Users/josepaulrichard@gmail.com/device_stream_new"

raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(inputPath)
)

print("Source Data Loaded")

print("Source Count :", raw_df.count())

print("Source Schema")

raw_df.printSchema()

display(raw_df)

# ============================================================
# BRONZE LAYER
# ============================================================

bronze_df = (
    raw_df
    .withColumn("ingestion_time", current_timestamp())
)

print("Bronze Count :", bronze_df.count())

display(bronze_df)

# ============================================================
# WRITE BRONZE TABLE
# ============================================================

bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("pikapika.default.device_bronze_jprg")

print("Bronze Table Created")

# ============================================================
# READ BRONZE TABLE
# ============================================================

bronze_table_df = spark.table("pikapika.default.device_bronze_jprg")

display(bronze_table_df)

# ============================================================
# SILVER LAYER
# ============================================================

silver_df = (
    bronze_table_df

    .filter(col("device_id").isNotNull())

    .withColumn(
        "temperature",
        col("temperature").cast("double")
    )

    .withColumn(
        "humidity",
        col("humidity").cast("double")
    )

    # IMPORTANT FIX HERE
    .withColumn(
        "event_timestamp",
        to_timestamp(col("timestamp"))
    )

    .dropDuplicates()
)

print("Silver Count :", silver_df.count())

display(silver_df)

# ============================================================
# WRITE SILVER TABLE
# ============================================================

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("pikapika.default.device_silver_jprg")

print("Silver Table Created")

# ============================================================
# READ SILVER TABLE
# ============================================================

silver_table_df = spark.table("pikapika.default.device_silver_jprg")

display(silver_table_df)

# ============================================================
# GOLD LAYER
# ============================================================

gold_df = (
    silver_table_df
    .groupBy("device_id")
    .agg(
        avg("temperature").alias("avg_temperature"),
        avg("humidity").alias("avg_humidity"),
        count("*").alias("total_records")
    )
)

print("Gold Count :", gold_df.count())

display(gold_df)

# ============================================================
# WRITE GOLD TABLE
# ============================================================

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("pikapika.default.device_gold_jprg")

print("Gold Table Created")

# ============================================================
# SHOW TABLES
# ============================================================

spark.sql("""
SHOW TABLES IN pikapika.default
""").show(truncate=False)

# ============================================================
# VALIDATION
# ============================================================

print("Bronze Table Count")

spark.sql("""
SELECT COUNT(*) AS bronze_count
FROM pikapika.default.device_bronze_jprg
""").show()

print("Silver Table Count")

spark.sql("""
SELECT COUNT(*) AS silver_count
FROM pikapika.default.device_silver_jprg
""").show()

print("Gold Table Count")

spark.sql("""
SELECT COUNT(*) AS gold_count
FROM pikapika.default.device_gold_jprg
""").show()

print("Pipeline Completed Successfully")

Old tables dropped
Source Data Loaded
Source Count : 183
Source Schema
root
 |-- device_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)



device_id,timestamp,temperature,humidity
1,2024-06-24T00:00:00Z,25.6,0.59
1,2024-06-24T00:00:10Z,25.9,0.56
1,2024-06-24T00:00:20Z,25.8,0.59
1,2024-06-24T00:00:30Z,25.6,0.58
1,2024-06-24T00:00:40Z,25.9,0.57
1,2024-06-24T00:00:50Z,26.0,0.59
1,2024-06-24T00:01:00Z,25.7,0.56
1,2024-06-24T00:01:10Z,26.0,0.57
1,2024-06-24T00:01:20Z,25.9,0.55
1,2024-06-24T00:01:30Z,26.0,0.58


Bronze Count : 183


device_id,timestamp,temperature,humidity,ingestion_time
1,2024-06-24T00:00:00Z,25.6,0.59,2026-05-22T10:52:29.236515Z
1,2024-06-24T00:00:10Z,25.9,0.56,2026-05-22T10:52:29.236515Z
1,2024-06-24T00:00:20Z,25.8,0.59,2026-05-22T10:52:29.236515Z
1,2024-06-24T00:00:30Z,25.6,0.58,2026-05-22T10:52:29.236515Z
1,2024-06-24T00:00:40Z,25.9,0.57,2026-05-22T10:52:29.236515Z
1,2024-06-24T00:00:50Z,26.0,0.59,2026-05-22T10:52:29.236515Z
1,2024-06-24T00:01:00Z,25.7,0.56,2026-05-22T10:52:29.236515Z
1,2024-06-24T00:01:10Z,26.0,0.57,2026-05-22T10:52:29.236515Z
1,2024-06-24T00:01:20Z,25.9,0.55,2026-05-22T10:52:29.236515Z
1,2024-06-24T00:01:30Z,26.0,0.58,2026-05-22T10:52:29.236515Z


Bronze Table Created


...

Silver Table Created


device_id,timestamp,temperature,humidity,ingestion_time,event_timestamp
1,2024-06-24T00:11:20Z,24.6,0.49,2026-05-22T10:52:30.490871Z,2024-06-24T00:11:20Z
1,2024-06-24T00:10:10Z,24.7,0.45,2026-05-22T10:52:30.490871Z,2024-06-24T00:10:10Z
1,2024-06-24T00:06:40Z,25.7,0.58,2026-05-22T10:52:30.490871Z,2024-06-24T00:06:40Z
1,2024-06-24T00:22:09Z,22.8,0.42,2026-05-22T10:52:30.490871Z,2024-06-24T00:22:09Z
1,2024-06-24T00:04:20Z,25.8,0.56,2026-05-22T10:52:30.490871Z,2024-06-24T00:04:20Z
1,2024-06-24T00:06:20Z,25.9,0.56,2026-05-22T10:52:30.490871Z,2024-06-24T00:06:20Z
1,2024-06-24T00:01:40Z,25.7,0.56,2026-05-22T10:52:30.490871Z,2024-06-24T00:01:40Z
1,2024-06-24T00:24:59Z,22.7,0.4,2026-05-22T10:52:30.490871Z,2024-06-24T00:24:59Z
1,2024-06-24T00:15:10Z,24.8,0.45,2026-05-22T10:52:30.490871Z,2024-06-24T00:15:10Z
1,2024-06-24T00:03:20Z,25.6,0.57,2026-05-22T10:52:30.490871Z,2024-06-24T00:03:20Z


Gold Count : 1


device_id,avg_temperature,avg_humidity,total_records
1,24.437569060773484,0.48994475138121496,181


Gold Table Created
+--------+--------------------+-----------+
|database|tableName           |isTemporary|
+--------+--------------------+-----------+
|default |bronze_tips         |false      |
|default |bronze_tips_tejas   |false      |
|default |covid_bronze        |false      |
|default |covid_bronze_ai     |false      |
|default |covid_bronze_bb     |false      |
|default |covid_bronze_gb     |false      |
|default |covid_bronze_gmani  |false      |
|default |covid_bronze_jeet   |false      |
|default |covid_bronze_jprg   |false      |
|default |covid_bronze_jv     |false      |
|default |covid_bronze_mas    |false      |
|default |covid_bronze_mk     |false      |
|default |covid_bronze_ms     |false      |
|default |covid_bronze_rb     |false      |
|default |covid_bronze_skannan|false      |
|default |covid_bronze_sp     |false      |
|default |covid_bronze_ss     |false      |
|default |covid_bronze_tm     |false      |
|default |covid_bronze_us10   |false      |
|default |cov